In [17]:
print("Training process for Random Forest...\n\n🛢️ Random Forest Pipeline: SMOTE -> Random Forest")
rf_pipeline = ImbPipeline([
    (
        "smote", 
        SMOTE(random_state=42)
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])
print("✅ Done!\n")

print("🔎 Random Forest hyperparameter search...")
rf_params = {
    "model__n_estimators": randint(150, 700),
    "model__max_depth": [10, 15, 20, 25, 30, None],
    "model__min_samples_split": randint(5, 50),
    "model__min_samples_leaf": randint(2, 15),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5]
}
print("✅ Done!\n")

print("🚀 Initializing cross-validation object...")
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
print("✅ Done!\n")

print("🎲 Initializing a search manager object...")
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter = 30,
    scoring="roc_auc",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
print("✅ Done!\n")

Training process for Random Forest...

🛢️ Random Forest Pipeline: SMOTE -> Random Forest
✅ Done!

🔎 Random Forest hyperparameter search...
✅ Done!

🚀 Initializing cross-validation object...
✅ Done!

🎲 Initializing a search manager object...
✅ Done!



In [18]:
print("🎓 Fitting the search manager...")
rf_search.fit(X_train_rf, y_train_rf)

print(f"Best hyperparameters for Random Forest model are:\n{rf_search.best_params_}")
print(f"The best score is:{rf_search.best_score_}\n")

best_rf = rf_search.best_estimator_
print("💾 Saving best performing Random Forest model...")

try:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    joblib.dump(best_rf, f"../models/random_forest_best_{timestamp}.joblib")
    joblib.dump(rf_search, f"../models/rf_random_search_{timestamp}.joblib")
    print("✅ Random Forest model with best performance score was successfully saved!")
    
except:
    print("❌ Something went wrong during the model serialization...Check Traceback call")

🎓 Fitting the search manager...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best hyperparameters for Random Forest model are:
{'model__max_depth': 20, 'model__max_features': 0.3, 'model__min_samples_leaf': 2, 'model__min_samples_split': 12, 'model__n_estimators': 568}
The best score is:0.9963681689443057

💾 Saving best performing Random Forest model...
✅ Random Forest model with best performance score was successfully saved!
[CV] END model__max_depth=30, model__max_features=0.5, model__min_samples_leaf=9, model__min_samples_split=28, model__n_estimators=280; total time= 4.3min
[CV] END model__max_depth=None, model__max_features=0.3, model__min_samples_leaf=4, model__min_samples_split=48, model__n_estimators=393; total time= 4.0min
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=44, model__n_estimators=537; total time= 1.5min
[CV] END model__max_depth=15, model__max_features=log2, model__min_samples_leaf=13,

In [21]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)
import pandas as pd
import numpy as np

In [22]:
print("🚀 Training baseline Isolation Forest...")

if_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    max_samples="auto",
    max_features=1.0,
    bootstrap=False,
    random_state=42,
    n_jobs=-1,
)

if_model.fit(X_train_if)
print("✅ Isolation Forest training finished.")

🚀 Training baseline Isolation Forest...


In [23]:
print("Helper function for evaluation")
def evaluate_if_period(
    model,
    X_period: pd.DataFrame,
    period_df: pd.DataFrame,
    labels_df: pd.DataFrame,
    period_name: str = "Validation",
):
    """
    Evaluate Isolation Forest on a temporal period.

    Parameters:
        model: fitted IsolationForest
        X_period: feature matrix for the whole period (all transactions)
        period_df: full period dataframe with txId and time_step
        labels_df: dataframe with columns ['txId', 'is_illicit'] for labeled transactions only
        period_name: name printed in the output
    """
    print(f"\n⚖️ Evaluation of Isolation Forest on {period_name} period...")

    # Raw model outputs for ALL transactions in the period
    raw_pred = model.predict(X_period)
    anomaly_score = -model.score_samples(X_period)

    predictions_df = pd.DataFrame({
        "txId": period_df["txId"].values,
        "anomaly_score": anomaly_score,
        "pred_label": (raw_pred == -1).astype(int),
    })

    # Keep only labeled transactions for metric calculation
    eval_df = predictions_df.merge(labels_df, on="txId", how="inner")

    y_true = eval_df["is_illicit"].astype(int)
    y_pred = eval_df["pred_label"].astype(int)
    y_score = eval_df["anomaly_score"].astype(float)

Helper function for evaluation
